In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:

endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )

c:\Users\Kashish\Downloads\UPCON26_PHP (5)\Agentic-AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model=ChatHuggingFace(llm=endpoint)

In [4]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    strike_rate: float
    centuries: int
    bpb: float
    boundary_percentage: float
    summary: str



In [5]:
def calculate_sr(state:BatsmanState)->BatsmanState:
    state['strike_rate']=state['runs']/state['balls']*100
    return {'strike_rate':state['strike_rate']}

In [6]:
def calculate_bpb(state:BatsmanState)->BatsmanState:
    state['bpb']=state['balls']/state['fours']+state['sixes']
    return {'bpb':state['bpb']}

In [7]:
def calculate_boundary_percentage(state:BatsmanState)->BatsmanState:
    state['boundary_percentage']= (((state['fours']*4)+(state['sixes']*6))/state['runs']) *100
    return {'boundary_percentage':state['boundary_percentage']}

In [8]:

def summary(state:BatsmanState)->BatsmanState:
    summary_str=f""" 
    Strike Rate - {state['strike_rate']}
    Balls per boundary - {state['bpb']}
    Boundary percentage - {state['boundary_percentage']}
    """

    state['summary']=summary_str
    return {'summary':summary_str}


In [9]:
graph=StateGraph(BatsmanState)

In [10]:
graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percentage',calculate_boundary_percentage)
graph.add_node('summary',summary)

In [11]:
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percentage')

graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percentage','summary')

graph.add_edge('summary',END)

workflow=graph.compile()


In [12]:
initial_state={'runs':100,'balls':100,'fours':10,'sixes':5}

result=workflow.invoke(initial_state)
print(result)


{'runs': 100, 'balls': 100, 'fours': 10, 'sixes': 5, 'strike_rate': 100.0, 'bpb': 15.0, 'boundary_percentage': 70.0, 'summary': ' \n    Strike Rate - 100.0\n    Balls per boundary - 15.0\n    Boundary percentage - 70.0\n    '}
